# Ticketmaster API Explorer
This notebook interacts with the Ticketmaster Discovery API to query events and venues in the UK.

## 1. Imports & Setup

In [ ]:
import requests
import os
import re
import time
from typing import Optional
from dotenv import load_dotenv
from pprint import pprint
import pandas as pd
# hi 

## 2. Configuration
Load the API key from the `.env` file and define base URL constants.

In [ ]:
load_dotenv()

API_KEY = os.getenv("TICKET_API")

if not API_KEY:
    raise ValueError("That API key is not in your env, check your env for the correct variable name.")

BASE_URL = "https://app.ticketmaster.com"

EVENTS_ENDPOINT = "/discovery/v2/events"
VENUES_ENDPOINT = "/discovery/v2/venues"

print("Config loaded successfully.")

## 3. Events
Query the Ticketmaster Discovery API for events in the UK.

In [ ]:
api_key = API_KEY
all_events = []
current_page = 0
total_pages = 1 

while current_page < total_pages:
    events_url = f"{BASE_URL}{EVENTS_ENDPOINT}.json?apikey={api_key}&countryCode=GB&size=200&page={current_page}"
    events_response = requests.get(events_url).json()
    
    page_events = events_response.get('_embedded', {}).get('events', [])
    all_events.extend(page_events)
    
    total_pages = events_response.get('page', {}).get('totalPages', 1)
    current_page += 1

print(f"Successfully collected {len(all_events)} events total!")





# events_url = f"{BASE_URL}{EVENTS_ENDPOINT}"

# payload_events = {
#     "apikey": API_KEY,
#     "size": "200",
#     "countryCode": "GB"
# }

# response_event = requests.get(events_url, params=payload_events)
# response_event.raise_for_status()

# data_event = response_event.json()

# print(f"Status: {response_event.status_code}")

In [ ]:
# original

# uk_events = []

# event_list = data_event['_embedded']['events']

# for event in event_list:
#     event_data = {
#         "event_id":   event.get("id"),
#         "name": event.get("name"),
#         "date": event.get("dates", {}).get("start", {}).get("localDate"),
#         "time": event.get("dates", {}).get("start", {}).get("localTime"),
#         "multi_day_event": event.get("dates", {}).get("spanMultipleDays"),
#         "legal_age_enforced": event.get("ageRestrictions", {}).get("legalAgeEnforced"),
#         "category_segment": event.get("classifications", {})[0].get("segment", {}).get("name"),
#         "category_genre": event.get("classifications", {})[0].get("genre", {}).get("name"),
#         "category_sub_genre": event.get("classifications", {})[0].get("subGenre", {}).get("name"),
#         "all_inclusive_pricing": event.get("ticketing", {}).get("allInclusivePricing", {}).get("enabled"),
#         "venue_id": event.get("_embedded", {}).get("venues", {})[0].get("id"),
#         "venue_name": event.get("_embedded", {}).get("venues", {})[0].get("name"),
#         "address": event.get("_embedded", {}).get("venues", {})[0].get("address", {}).get("line1"),
#         "postcode": event.get("_embedded", {}).get("venues", {})[0].get("postalCode"),
#         "city": event.get("_embedded", {}).get("venues", {})[0].get("city", {}).get("name"),
#         "longitude": event.get("_embedded", {}).get("venues", {})[0].get("location", {}).get("longitude"),
#         "latitude": event.get("_embedded", {}).get("venues", {})[0].get("location", {}).get("latitude"),
#         "markets": event.get("_embedded", {}).get("venues", {})[0].get("markets")[0].get("name"),
#         "markets_id": event.get("_embedded", {}).get("venues", {})[0].get("markets")[0].get("id"),
#         "url": event.get("url"),

#     }

#     uk_events.append(event_data)

In [ ]:
# kiro

uk_events = []

for event in all_events:
    class_list = event.get("classifications") or []
    cls = class_list[0] if class_list else {}

    venue_list = event.get("_embedded", {}).get("venues") or []
    vn = venue_list[0] if venue_list else {}

    market_list = vn.get("markets") or []
    mkt = market_list[0] if market_list else {}
    event_data = {
        "event_id": event.get("id"),
        "name": event.get("name"),
        "date": event.get("dates", {}).get("start", {}).get("localDate"),
        "time": event.get("dates", {}).get("start", {}).get("localTime"),
        "multi_day_event": event.get("dates", {}).get("spanMultipleDays"),
        "legal_age_enforced": event.get("ageRestrictions", {}).get("legalAgeEnforced"),

        "category_segment": cls.get("segment", {}).get("name"),
        "category_genre": cls.get("genre", {}).get("name"),
        "category_sub_genre": cls.get("subGenre", {}).get("name"),

        "all_inclusive_pricing": event.get("ticketing", {}).get("allInclusivePricing", {}).get("enabled"),

        "venue_id": vn.get("id"),
        "venue_name": vn.get("name"),
        "address": vn.get("address", {}).get("line1"),
        "postcode": vn.get("postalCode"),
        "city": vn.get("city", {}).get("name"),
        "longitude": vn.get("location", {}).get("longitude"),
        "latitude": vn.get("location", {}).get("latitude"),

        "markets": mkt.get("name"),
        "markets_id": mkt.get("id"),
        "url": event.get("url")
    }

    uk_events.append(event_data)

In [ ]:
uk_events[0]

In [ ]:
print(len(uk_events))

In [ ]:
# implement pydantic validation

In [ ]:
df_events = pd.DataFrame(uk_events)
df_events.head()

In [ ]:
df_events.shape

## 4. Venues
Query the Ticketmaster Discovery API for venues in the UK and extract key fields.

In [ ]:
# venues_url = f"{BASE_URL}{VENUES_ENDPOINT}"

# payload_venues = {
#     "apikey": API_KEY,
#     "countryCode": "GB",
#     "size": "200"
# }

# response_venue = requests.get(venues_url, params=payload_venues)
# response_venue.raise_for_status()

# data_venues = response_venue.json()

# print(f"Status: {response_venue.status_code}")


all_venues = []
current_page = 0
total_pages = 1 

while current_page < total_pages:
    venues_url = f"{BASE_URL}{VENUES_ENDPOINT}.json?apikey={api_key}&countryCode=GB&size=200&page={current_page}"
    venues_response = requests.get(venues_url).json()
    
    page_venues = venues_response.get('_embedded', {}).get('venues', [])
    all_venues.extend(page_venues)
    
    total_pages = venues_response.get('page', {}).get('totalPages', 1)
    current_page += 1

print(f"Successfully collected {len(all_venues)} venues total!")


In [ ]:
# uk_venues = []

# for venue in data_venues["_embedded"]["venues"]:
#     venue_data = {
#         "id":         venue.get("id"),
#         "name":       venue.get("name"),
#         "city":       venue.get("city", {}).get("name"),
#         "country":    venue.get("country", {}).get("name"),
#         "longitude":  venue.get("location", {}).get("longitude"),
#         "latitude":   venue.get("location", {}).get("latitude"),
#         "postcode": venue.get("postalCode")
#     }
#     uk_venues.append(venue_data)


uk_venues = []

for venue in all_venues:
    venue_data = {
        "id":        venue.get("id"),
        "name":      venue.get("name"),
        "city":      venue.get("city", {}).get("name"),
        "country":   venue.get("country", {}).get("name"),
        "longitude": venue.get("location", {}).get("longitude"),
        "latitude":  venue.get("location", {}).get("latitude"),
        "postcode":  venue.get("postalCode")
    }
    uk_venues.append(venue_data)

In [ ]:
# Preview the first venue
pprint(uk_venues[0])

In [ ]:
df_venues = pd.DataFrame(uk_venues)
df_venues.head()

## 5. Fetch Venue Capacity
Since the Ticketmaster API does not expose capacity data, we fetch it from Wikipedia
using the MediaWiki API. The function searches for each venue by name, pulls the raw
wikitext (infobox) and plain-text extract, then uses regex to extract the capacity value.

In [ ]:
WIKI_API = "https://en.wikipedia.org/w/api.php"

# Patterns ordered most-specific (infobox) → least-specific (prose)
CAPACITY_PATTERNS = [
    r"\|\s*capacity\s*=\s*([\d,]+)",          # | capacity = 62,062
    r"\|\s*seating_capacity\s*=\s*([\d,]+)",  # | seating_capacity = …
    r"capacity\s*[:=]\s*([\d,]+)",             # capacity: 20,000
    r"([\d,]+)\s*-seat",                        # 20,000-seat arena
    r"seats?\s+([\d,]+)",                       # seats 20,000
    r"([\d,]+)\s*capacity",                     # 20,000 capacity
    r"holds?\s+([\d,]+)",                       # holds 20,000
]


def _search_wikipedia(query: str) -> list:
    """Return up to 3 Wikipedia page titles matching the query."""
    resp = requests.get(WIKI_API, params={
        "action":   "query",
        "format":   "json",
        "list":     "search",
        "srsearch": query,
        "srlimit":  3,
    }, timeout=10)
    resp.raise_for_status()
    return resp.json().get("query", {}).get("search", [])


def _get_wikitext(page_title: str) -> str:
    """
    Fetch the raw wikitext for a page (needed to read the infobox).
    Uses a dedicated revisions-only request so the slot content is always returned.
    """
    resp = requests.get(WIKI_API, params={
        "action":  "query",
        "format":  "json",
        "titles":  page_title,
        "prop":    "revisions",
        "rvprop":  "content",
        "rvslots": "main",
    }, timeout=10)
    resp.raise_for_status()
    pages = resp.json().get("query", {}).get("pages", {})
    for page in pages.values():
        revisions = page.get("revisions", [])
        if revisions:
            return revisions[0].get("slots", {}).get("main", {}).get("*", "")
    return ""


def _get_extract(page_title: str) -> str:
    """Fetch the plain-text article extract for a page."""
    resp = requests.get(WIKI_API, params={
        "action":     "query",
        "format":     "json",
        "titles":     page_title,
        "prop":       "extracts",
        "explaintext": True,
        "exintro":    False,   # get full article, not just intro
    }, timeout=10)
    resp.raise_for_status()
    pages = resp.json().get("query", {}).get("pages", {})
    for page in pages.values():
        return page.get("extract", "")
    return ""


def _extract_capacity_from_text(text: str) -> Optional[int]:
    """Apply all capacity patterns to a block of text. Returns first valid hit."""
    for pattern in CAPACITY_PATTERNS:
        for raw_match in re.findall(pattern, text, re.IGNORECASE):
            clean = raw_match.replace(",", "").strip()
            num = re.search(r"\d+", clean)
            if num:
                capacity = int(num.group())
                if 50 <= capacity <= 200_000:
                    return capacity
    return None


def fetch_capacity(venue_name: str, city: str = None) -> Optional[int]:
    """
    Fetch venue capacity from Wikipedia.

    Makes separate requests for wikitext (infobox) and plain-text extract
    so neither call interferes with the other.

    Args:
        venue_name: Name of the venue.
        city:       City name — helps Wikipedia disambiguation.

    Returns:
        Capacity as int if found, None otherwise.
    """
    try:
        search_query = f"{venue_name} {city}" if city else venue_name
        results = _search_wikipedia(search_query)

        if not results:
            return None

        for result in results:
            page_title = result["title"]

            # 1️⃣  Try raw wikitext first — infobox is the most reliable source
            wikitext = _get_wikitext(page_title)
            if wikitext:
                cap = _extract_capacity_from_text(wikitext)
                if cap:
                    return cap

            # 2️⃣  Fall back to the plain-text article extract
            extract = _get_extract(page_title)
            if extract:
                cap = _extract_capacity_from_text(extract)
                if cap:
                    return cap

        return None

    except Exception as exc:
        print(f"  [WARN] Could not fetch capacity for '{venue_name}': {exc}")
        return None

In [ ]:
# ── Debug helper ─────────────────────────────────────────────────────────────
# Use this cell to inspect exactly what Wikipedia returns for a single venue.
# Useful when fetch_capacity returns None and you want to understand why.

def debug_venue(venue_name: str, city: str = None):
    query = f"{venue_name} {city}" if city else venue_name
    results = _search_wikipedia(query)
    print(f"Search results for '{query}':")
    for r in results:
        print(f"  → {r['title']}")

    if results:
        title = results[0]["title"]
        print(f"\nWikitext snippet for '{title}':")
        wikitext = _get_wikitext(title)
        # Print the first 1500 chars — enough to see the infobox
        print(wikitext[:1500] if wikitext else "  (empty)")

# Example — swap in any venue that is returning None
debug_venue("Wembley Stadium", "London")

In [ ]:
# Smoke-test on well-known venues
test_venues = [
    ("Wembley Stadium",           "London"),
    ("Tottenham Hotspur Stadium", "London"),
    ("O2 Arena",                  "London"),
    ("Old Trafford",              "Manchester"),
]

print("Venue capacity smoke-test\n" + "-" * 45)
for name, city in test_venues:
    cap = fetch_capacity(name, city)
    if cap:
        print(f"  {name:<35} {cap:>8,}")
    else:
        print(f"  {name:<35} {'not found':>8}")

In [ ]:
def fetch_capacities_batch(
    venues_df: pd.DataFrame,
    rate_limit_delay: float = 0.2
) -> pd.DataFrame:
    """
    Fetch capacities for every venue in the DataFrame.

    Deduplicates by venue name so each unique venue is only looked up once,
    then maps the result back to all rows.

    Args:
        venues_df:         DataFrame with at least 'name' and 'city' columns.
        rate_limit_delay:  Seconds to wait between Wikipedia requests.

    Returns:
        A copy of venues_df with a new 'capacity' column.
    """
    # Build a deduplicated lookup so we don't hammer Wikipedia for repeated venues
    unique_venues = venues_df[["name", "city"]].drop_duplicates(subset="name")
    capacity_map  = {}  # name -> capacity

    total = len(unique_venues)
    for i, (_, row) in enumerate(unique_venues.iterrows(), start=1):
        name = row["name"]
        city = row.get("city")
        print(f"[{i:>3}/{total}] {name} ...", end=" ", flush=True)

        cap = fetch_capacity(name, city)
        capacity_map[name] = cap

        print(f"{cap:,}" if cap else "not found")
        time.sleep(rate_limit_delay)

    result = venues_df.copy()
    result["capacity"] = result["name"].map(capacity_map)
    return result

In [ ]:
# Run the batch fetch across all venues
# ~200 unique venues at 0.2 s delay takes roughly 40–60 seconds
df_venues_with_capacity = fetch_capacities_batch(df_venues, rate_limit_delay=0.2)
df_venues_with_capacity.head(10)

In [ ]:
# Summary stats
found    = df_venues_with_capacity["capacity"].notna().sum()
not_found = df_venues_with_capacity["capacity"].isna().sum()
total    = len(df_venues_with_capacity)

print(f"Total venues   : {total}")
print(f"Capacity found : {found}  ({found/total*100:.1f}%)")
print(f"Not found      : {not_found}  ({not_found/total*100:.1f}%)")

print("\nTop 10 venues by capacity:")
df_venues_with_capacity.nlargest(10, "capacity")[["name", "city", "capacity"]]